In [ ]:
# Gold Aggregation — Notebook de respaldo
# Genera post_counts_by_user desde Silver

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, when, current_timestamp, coalesce, lit

spark = (
    SparkSession.builder
    .appName("gold_notebook")
    .master("spark://spark-master:7077")
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "software.amazon.awssdk:bundle:2.24.8,"
        "software.amazon.awssdk:url-connection-client:2.24.8,"
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
        "org.projectnessie.nessie-integrations:nessie-spark-extensions-3.5_2.12:0.77.1")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    .config("spark.sql.defaultCatalog", "nessie")
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1")
    .config("spark.sql.catalog.nessie.ref", "main")
    .config("spark.sql.catalog.nessie.authentication.type", "NONE")
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse/")
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true")
    .config("spark.sql.catalog.nessie.s3.region", "us-east-1")
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin")
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .getOrCreate()
)
print(f"Spark: {spark.version}")

In [ ]:
# Leer Silver
df_posts = spark.sql("SELECT * FROM nessie.silver.posts_hist")
df_users = spark.sql("SELECT * FROM nessie.silver.users_hist")
print(f"posts_hist: {df_posts.count()} | users_hist: {df_users.count()}")

In [ ]:
# Agregacion Gold
df_agg = (
    df_posts.groupBy("OwnerUserId").agg(
        count("*").alias("total_posts"),
        spark_sum(when(col("PostTypeId") == "1", 1).otherwise(0)).alias("total_preguntas"),
        spark_sum(when(col("PostTypeId") == "2", 1).otherwise(0)).alias("total_respuestas"),
        spark_sum(coalesce(col("Score").cast("int"), lit(0))).alias("total_score"),
        spark_sum(coalesce(col("ViewCount").cast("int"), lit(0))).alias("total_views"),
        spark_sum(coalesce(col("CommentCount").cast("int"), lit(0))).alias("total_comments"),
        spark_sum(coalesce(col("AnswerCount").cast("int"), lit(0))).alias("total_answers_received"),
    )
)
df_gold = (
    df_agg.join(
        df_users.select(col("Id").alias("UserId"), "DisplayName", "Reputation", "Location", "UpVotes", "DownVotes", "Views"),
        df_agg.OwnerUserId == col("UserId"), "left"
    ).drop("UserId").withColumn("fecha_cargue", current_timestamp())
)
print(f"Gold filas: {df_gold.count()}")
df_gold.show(5, truncate=False)

In [ ]:
# Escribir Gold con MERGE
spark.sql("CREATE NAMESPACE IF NOT EXISTS nessie.gold")

try:
    spark.sql("SELECT 1 FROM nessie.gold.post_counts_by_user LIMIT 1")
    df_gold.createOrReplaceTempView("gold_in")
    spark.sql('''
        MERGE INTO nessie.gold.post_counts_by_user AS t
        USING gold_in AS s ON t.OwnerUserId = s.OwnerUserId
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    ''')
    print("MERGE Gold completado")
except:
    df_gold.writeTo("nessie.gold.post_counts_by_user").create()
    print("Tabla Gold creada")

spark.sql("SELECT COUNT(*) as total FROM nessie.gold.post_counts_by_user").show()
spark.stop()